# Phase 0: baseline WER on Vistaar benchmarks (Kaggle, 2× T4)

Setup, once:
1. Right sidebar → **Session options**: Accelerator **GPU T4 x2**, **Internet on** (both need a phone-verified account).
2. **Add-ons → Secrets**: add `HF_TOKEN` (Hugging Face **write** token) and tick it for this notebook.
3. **Run all** (or *Save Version → Save & Run All* to run in the background, up to 12 h).

GPU 0 runs whisper-small then whisper-large-v3-turbo; GPU 1 runs IndicWhisper-medium. Every result uploads to the Hub as it lands, so a rerun resumes.

In [ ]:
RESULTS_REPO = "akashkuushwahaa/indic-whisper-results"  # private dataset, created if missing
WORK = "/tmp/iw"

import os, shutil, subprocess

def run(cmd):
    """Like `!cmd`, but raises when the command fails."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")  # fails if the secret isn't attached
if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU: Session options -> Accelerator -> GPU T4 x2 (needs a phone-verified account).")
run("nvidia-smi --query-gpu=index,name,memory.total --format=csv")
run("df -h /tmp | tail -1")
run("pip install -q --upgrade 'git+https://github.com/akashkuushwahaa/indic-whisper'")  # fails if Internet is off
run("hf auth whoami")

In [ ]:
# Build the gu/hi benchmarks once (~3 GB) before the two GPU jobs start.
run(f"iw-bench-build --out {WORK}/bench")

In [ ]:
jobs = {
    0: "whisper-small whisper-large-v3-turbo",
    1: "indicwhisper-medium",
}
procs = {}
for gpu, models in jobs.items():
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    log = open(f"/kaggle/working/eval-gpu{gpu}.log", "w")
    procs[gpu] = subprocess.Popen(
        f"iw-eval-job --results-repo {RESULTS_REPO} --work {WORK} --device cuda --only {models}",
        shell=True, env=env, stdout=log, stderr=subprocess.STDOUT)

import time
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    for gpu in jobs:
        lines = [l for l in open(f"/kaggle/working/eval-gpu{gpu}.log") if "WER" in l or "Error" in l]
        print(f"gpu{gpu}:", lines[-1].strip() if lines else "(starting)", flush=True)
codes = {g: p.returncode for g, p in procs.items()}
print("exit codes:", codes)
if any(codes.values()):
    raise SystemExit("a job failed; see /kaggle/working/eval-gpu*.log")

In [ ]:
run(f"iw-eval-report --out {WORK}/results")